[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cCoolBoox/IFCD107-alumnado/blob/main/sprints/S12/2_Practica/para_casa/S12_01_series_temporales.ipynb)

# S12 · 01 — Series temporales: baseline, descomposición y ARIMA/SARIMAX

Sprint 12 · UD 4.7

**Contexto TurisData.** La dirección de TurisData quiere saber cuánta ocupación tendrá el alojamiento la próxima semana para organizar personal y compras. Tenemos la ocupación diaria (%) de 2023 a 2025.

**Este cuaderno tiene 4 ejercicios y un mini-reto.** Los ejercicios 1, 2 y 4 son los mismos que los del cuaderno «en clase» (allí, del 1 al 3).

**Cómo usar este cuaderno** (tiene dos partes):

- **Parte A · Ejecutar** (aquí abajo): de arriba abajo, cada celda de código lleva encima su título con un símbolo:
    - ⚙️ **Preparación**: se ejecuta sola, pulsa ▶ y sigue.
    - 👀 **Ejemplo**: pulsa ▶, mira el resultado y responde las preguntas 🧠 (no hay que escribir código). Las respuestas están en la Parte B.
    - ✏️ **Ejercicio**: tu turno. Cambia los `...` y pulsa ▶. Si sale **✔**, responde las preguntas 🧠 y sigues. Para comprobar tu código, tienes la solución de referencia ✅ en la Parte B.
- **Parte B · Explicación** (al final): para cada celda, **qué hace**, **qué resultado debe salir** y, en los ejemplos, las **respuestas** a las preguntas 🧠 y, en los ejercicios, la **solución de referencia** ✅ para comprobar tu código. Ábrela cuando quieras entender una celda, comprobar tus respuestas o no te salga. Usa el índice de la izquierda (☰) para saltar.

# Parte A · Ejecutar

### ⚙️ A1 · PREPARACIÓN — se ejecuta sola

Pulsa ▶ y sigue. No hace falta leerla.

In [ ]:
#@title A1 · ▶ Ejecuta esta celda primero (no hace falta leerla)
import os, urllib.request

BASE_URL = "https://raw.githubusercontent.com/cCoolBoox/IFCD107-alumnado/main/datos/"

def dato(nombre):
    """Devuelve la ruta local de un archivo de datos del curso (lo descarga si no está)."""
    for carpeta in ("datos", "../datos", "../../datos", "../../../datos"):
        ruta = os.path.join(carpeta, nombre)
        if os.path.exists(ruta):
            return ruta
    os.makedirs("datos", exist_ok=True)
    ruta = os.path.join("datos", nombre)
    urllib.request.urlretrieve(BASE_URL + nombre, ruta)
    return ruta

import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.statespace.sarimax import SARIMAX

warnings.filterwarnings("ignore")
SEMILLA = 42
np.random.seed(SEMILLA)
plt.rcParams.update({"figure.figsize": (10, 3.5), "axes.grid": True, "axes.spines.top": False, "axes.spines.right": False})
print("Listo ✔")

### 👀 A2 · EJEMPLO — Cargar la ocupación diaria

Pulsa ▶ y mira lo que sale. **No hay que escribir nada.**

🧠 **Autoejercicio:** mira el resultado y responde (respuestas en la Parte B, A2).

1. ¿Cuántos días y columnas hay?
2. ¿Para qué sirve `set_index("fecha")`?
3. ¿Qué indica `freq = "D"`?

In [ ]:
ocup = pd.read_csv(dato("ocupacion_diaria.csv"), parse_dates=["fecha"]).set_index("fecha")
ocup.index.freq = "D"
print(ocup.shape)
ocup.head()

### 👀 A3 · EJEMPLO — Dibujar la serie y sus patrones semanal y anual

Pulsa ▶ y mira lo que sale. **No hay que escribir nada.**

🧠 **Autoejercicio:** mira el resultado y responde (respuestas en la Parte B, A3).

1. ¿Qué días de la semana tienen más ocupación?
2. ¿Qué mes tiene la ocupación más baja y cuál la más alta?
3. ¿Qué son la tendencia, la estacionalidad y el ruido?

In [ ]:
fig, ax = plt.subplots()
ocup["ocupacion_pct"].plot(ax=ax, lw=0.8)
ax.set_title("Ocupación diaria del alojamiento (2023-2025)")
ax.set_xlabel("Fecha"); ax.set_ylabel("Ocupación (%)")
plt.show()

# Patrón semanal y anual, promediando
por_dia = ocup["ocupacion_pct"].groupby(ocup.index.dayofweek).mean()
por_mes = ocup["ocupacion_pct"].groupby(ocup.index.month).mean()
fig, ejes = plt.subplots(1, 2, figsize=(11, 3.5))
por_dia.index = ["lun", "mar", "mié", "jue", "vie", "sáb", "dom"]
por_dia.plot.bar(ax=ejes[0], color="#3b7ea1"); ejes[0].set_title("Ocupación media por día de la semana")
por_mes.plot.bar(ax=ejes[1], color="#e08e45"); ejes[1].set_title("Ocupación media por mes")
ejes[0].set_ylabel("Ocupación (%)"); ejes[1].set_xlabel("Mes")
plt.tight_layout(); plt.show()

### ✏️ A4 · EJERCICIO 1 de 4 — Día de más ocupación y mes de menos

**Tu turno:** ¿Qué día de la semana tiene más ocupación media y qué mes tiene menos? Guarda el nombre del día en `dia_max` (texto, p. ej. `"sáb"`) y el número de mes en `mes_min` (entero de 1 a 12).

**Debe salir:** ✔ Ejercicio 1 correcto

Cambia los `...` y pulsa ▶.

🧠 **Autoejercicio:** cuando salga ✔, responde (respuestas en la Parte B, A4).

1. ¿Qué valen `dia_max` y `mes_min`?
2. ¿Qué diferencia hay entre `max()` e `idxmax()`?
3. ¿Por qué es útil conocer el patrón semanal?

✅ ¿Dudas de tu código? Compáralo con la solución de referencia en la Parte B, A4, **después de intentarlo**.

In [ ]:
# pista: usa idxmax() sobre por_dia y idxmin() sobre por_mes
dia_max = ...
mes_min = ...

assert dia_max in ("vie", "sáb", "dom"), "Revisa dia_max: el fin de semana es el que más ocupa."
assert mes_min == 9, "Revisa mes_min: mira las barras de ocupación por mes y quédate con la más baja."
print("✔ Ejercicio 1 correcto")

### 👀 A5 · EJEMPLO — Partir entrenamiento y prueba por fecha

Pulsa ▶ y mira lo que sale. **No hay que escribir nada.**

🧠 **Autoejercicio:** mira el resultado y responde (respuestas en la Parte B, A5).

1. ¿Cuántos días hay en entrenamiento y en prueba?
2. ¿Por qué no se parte al azar?
3. ¿Qué usa `iloc[-DIAS_TEST:]`?

In [ ]:
DIAS_TEST = 91
serie = ocup["ocupacion_pct"]
entrenamiento, prueba = serie.iloc[:-DIAS_TEST], serie.iloc[-DIAS_TEST:]
print("Entrenamiento:", entrenamiento.index.min().date(), "→", entrenamiento.index.max().date(), len(entrenamiento))
print("Prueba       :", prueba.index.min().date(), "→", prueba.index.max().date(), len(prueba))

### 👀 A6 · EJEMPLO — Los dos primeros baselines (ingenuo y estacional)

Pulsa ▶ y mira lo que sale. **No hay que escribir nada.**

🧠 **Autoejercicio:** mira el resultado y responde (respuestas en la Parte B, A6).

1. ¿Qué MAE tienen el baseline ingenuo y el estacional?
2. ¿Cómo se imita el uso real en `evaluar_semanal`?
3. ¿Por qué se empieza con baselines?

In [ ]:
HORIZONTE = 7
origenes = list(range(len(entrenamiento), len(serie), HORIZONTE))   # posiciones donde empieza cada semana de test
origenes = [o for o in origenes if o + HORIZONTE <= len(serie)]

def mae(real, pred):
    return float(np.mean(np.abs(np.asarray(real) - np.asarray(pred))))

def evaluar_semanal(predictor):
    """predictor(historia: Series, origen: int) -> array de 7 valores. Devuelve (MAE, predicciones concatenadas)."""
    preds, reales = [], []
    for o in origenes:
        preds.append(np.asarray(predictor(serie.iloc[:o], o)))
        reales.append(serie.iloc[o:o + HORIZONTE].values)
    preds, reales = np.concatenate(preds), np.concatenate(reales)
    return mae(reales, preds), preds

def pred_ingenuo(historia, o):
    return np.repeat(historia.iloc[-1], HORIZONTE)

def pred_estacional(historia, o):
    return historia.iloc[-7:].values

mae_ing, p_ing = evaluar_semanal(pred_ingenuo)
mae_est, p_est = evaluar_semanal(pred_estacional)
print(f"MAE ingenuo    : {mae_ing:.2f} puntos")
print(f"MAE estacional : {mae_est:.2f} puntos")

### ✏️ A7 · EJERCICIO 2 de 4 — Baseline: media de las últimas 4 semanas

**Tu turno:** Un tercer baseline: la **media de las últimas 4 semanas** para cada día de la semana (más estable que copiar solo la última semana). Completa `pred_media4`: debe devolver 7 valores, uno por día, promediando `historia.iloc[-7]`, `historia.iloc[-14]`, `historia.iloc[-21]` y `historia.iloc[-28]`, y así para cada día de la semana siguiente. Después se calcula `mae_media4`.

**Debe salir:** `MAE media 4 semanas: 4.45` y ✔ Ejercicio 2 correcto

Cambia los `...` y pulsa ▶.

🧠 **Autoejercicio:** cuando salga ✔, responde (respuestas en la Parte B, A7).

1. ¿Cuánto vale el MAE de la media de 4 semanas y cómo se compara?
2. ¿Qué hace `reshape(4, 7)` y por qué `axis=0`?
3. ¿Por qué promediar 4 semanas en vez de copiar solo la última?

✅ ¿Dudas de tu código? Compáralo con la solución de referencia en la Parte B, A7, **después de intentarlo**.

In [ ]:
def pred_media4(historia, o):
    # pista: apila las últimas 28 observaciones en una matriz de 4 filas x 7 columnas y haz la media por columnas
    ...

mae_media4, p_media4 = evaluar_semanal(pred_media4)
print(f"MAE media 4 semanas: {mae_media4:.2f}")
assert len(pred_media4(serie.iloc[:400], 400)) == 7, "pred_media4 debe devolver 7 valores."
assert mae_media4 < mae_ing, "La media de 4 semanas debería batir al ingenuo; revisa el reshape (4, 7) y el eje de la media."
print("✔ Ejercicio 2 correcto")

### 👀 A8 · EJEMPLO — Descomponer la serie de entrenamiento

Pulsa ▶ y mira lo que sale. **No hay que escribir nada.**

🧠 **Autoejercicio:** mira el resultado y responde (respuestas en la Parte B, A8).

1. ¿Cuánto vale el efecto semanal de un fin de semana?
2. ¿Y entre semana?
3. ¿Qué representa el «resto»?

In [ ]:
desc = seasonal_decompose(entrenamiento, model="additive", period=7)
fig, ejes = plt.subplots(4, 1, figsize=(10, 8), sharex=True)
for ax, (nombre, s) in zip(ejes, [("Observada", desc.observed), ("Tendencia", desc.trend),
                                   ("Estacionalidad semanal", desc.seasonal), ("Resto", desc.resid)]):
    ax.plot(s, lw=0.8); ax.set_ylabel(nombre)
ejes[0].set_title("Descomposición aditiva (periodo = 7 días)")
plt.tight_layout(); plt.show()
print("Efecto semanal estimado (puntos sobre/bajo la media):")
print(desc.seasonal.iloc[:7].round(1).to_string())

### ✏️ A9 · EJERCICIO 3 de 4 — Amplitud semanal y resto

**Tu turno:** ¿Cuánta de la variación se explica por la estacionalidad semanal? Calcula en `amplitud_semanal` la diferencia entre el mayor y el menor efecto estacional (`desc.seasonal.max() - desc.seasonal.min()`), redondeada a 1 decimal, y guarda en `resto_std` la desviación típica del resto (sin NaN).

**Debe salir:** ✔ Ejercicio 3 correcto (`amplitud_semanal` = 9.4 y `resto_std` ≈ 3.67)

Cambia los `...` y pulsa ▶.

🧠 **Autoejercicio:** cuando salga ✔, responde (respuestas en la Parte B, A9).

1. ¿Cuánto valen `amplitud_semanal` y `resto_std`?
2. ¿Por qué se usa `dropna()` con el resto?
3. ¿Qué indica que `resto_std` (3,67) sea mucho menor que la desviación de la serie (9,61)?

✅ ¿Dudas de tu código? Compáralo con la solución de referencia en la Parte B, A9, **después de intentarlo**.

In [ ]:
# pista: max - min del componente seasonal; std() ignora NaN en pandas si usas dropna()
amplitud_semanal = ...
resto_std = ...

assert 5 < amplitud_semanal < 15, "amplitud_semanal debería rondar la brecha entre fin de semana y entre semana (~9 puntos)."
assert 0 < resto_std < serie.std(), "resto_std debe ser menor que la desviación de la serie original."
print("✔ Ejercicio 3 correcto")

### 👀 A10 · EJEMPLO — Ajustar un modelo SARIMAX con festivos y Fourier

Pulsa ▶ y mira lo que sale. **No hay que escribir nada.**

🧠 **Autoejercicio:** mira el resultado y responde (respuestas en la Parte B, A10).

1. ¿Cuánto suma un festivo según el modelo?
2. ¿Por qué se usan términos de Fourier en vez de una estacionalidad anual de 365 días?
3. ¿Por qué no se usa la temperatura como variable?

In [ ]:
def exogenas(indice):
    dia_anio = indice.dayofyear.values
    X = pd.DataFrame(index=indice)
    for k in (1, 2):
        X[f"sin{k}"] = np.sin(2 * np.pi * k * dia_anio / 365.25)
        X[f"cos{k}"] = np.cos(2 * np.pi * k * dia_anio / 365.25)
    X["festivo"] = ocup.loc[indice, "festivo"].values
    return X

X_todo = exogenas(serie.index)
X_ent = X_todo.iloc[:-DIAS_TEST]

modelo = SARIMAX(entrenamiento, exog=X_ent, order=(1, 0, 1), seasonal_order=(1, 1, 1, 7),
                 enforce_stationarity=False, enforce_invertibility=False)
res = modelo.fit(disp=False, maxiter=200)
print(res.summary().tables[1])

### 👀 A11 · EJEMPLO — Predecir semana a semana con SARIMAX

Pulsa ▶ y mira lo que sale. **No hay que escribir nada.**

🧠 **Autoejercicio:** mira el resultado y responde (respuestas en la Parte B, A11).

1. ¿Cuánto vale el MAE de SARIMAX y cómo se compara con los baselines?
2. ¿Por qué no se reajusta el modelo cada semana?
3. ¿Qué horizonte de previsión se usa?

In [ ]:
def pred_sarimax(historia, o):
    r = res.apply(historia, exog=X_todo.iloc[:o], refit=False)
    return r.forecast(HORIZONTE, exog=X_todo.iloc[o:o + HORIZONTE]).values

mae_sar, p_sar = evaluar_semanal(pred_sarimax)
print(f"MAE SARIMAX: {mae_sar:.2f}   (estacional ingenuo: {mae_est:.2f}, media 4 sem.: {mae_media4:.2f})")

### ✏️ A12 · EJERCICIO 4 de 4 — SARIMAX sin exógenas

**Tu turno:** Prueba una segunda especificación **sin** los festivos ni Fourier (solo `order=(1,0,1)`, `seasonal_order=(1,1,1,7)`) y compara. Guarda su MAE en `mae_sin_exog`. Pista: `SARIMAX(entrenamiento, order=..., seasonal_order=...)`, y en `res2.apply(historia, refit=False)` no hay `exog`.

**Debe salir:** `MAE SARIMAX sin exógenas: 3.93  |  con exógenas: 3.18` y ✔ Ejercicio 4 correcto

Cambia los `...` y pulsa ▶.

🧠 **Autoejercicio:** cuando salga ✔, responde (respuestas en la Parte B, A12).

1. ¿Cuánto vale el MAE sin exógenas y cuánto con ellas?
2. ¿Qué aportan los festivos y los términos de Fourier?
3. ¿Gana el modelo sin exógenas a los baselines?

✅ ¿Dudas de tu código? Compáralo con la solución de referencia en la Parte B, A12, **después de intentarlo**.

In [ ]:
# pista: ajusta res2 sin exog y repite el bucle semanal con res2.apply(historia, refit=False).forecast(HORIZONTE)
res2 = ...
mae_sin_exog = ...

assert 0 < mae_sin_exog < 15, "mae_sin_exog debe ser un MAE razonable (en puntos de ocupación)."
print(f"MAE SARIMAX sin exógenas: {mae_sin_exog:.2f}  |  con exógenas: {mae_sar:.2f}")
print("✔ Ejercicio 4 correcto")

### 👀 A13 · EJEMPLO — Tabla comparativa y gráfico de las previsiones

Pulsa ▶ y mira lo que sale. **No hay que escribir nada.**

🧠 **Autoejercicio:** mira el resultado y responde (respuestas en la Parte B, A13).

1. ¿Qué modelo tiene menos error y cuál más?
2. ¿Cuándo sería mejor quedarse con un baseline?
3. ¿Por qué se dibuja también la serie real?

In [ ]:
comparativa = pd.Series({"Ingenuo": mae_ing, "Estacional (t-7)": mae_est, "Media 4 semanas": mae_media4,
                         "SARIMAX sin exógenas": mae_sin_exog, "SARIMAX + festivos + Fourier": mae_sar}).sort_values()
print(comparativa.round(2).to_string())

fig, ax = plt.subplots()
ax.plot(prueba.index, prueba.values, label="Real", color="black", lw=1.2)
n = len(p_sar); ind = serie.index[origenes[0]:origenes[0] + n]
ax.plot(ind, p_est, label="Estacional t-7", alpha=.8)
ax.plot(ind, p_sar, label="SARIMAX", alpha=.9)
ax.set_title("Previsión a 7 días en las últimas 13 semanas"); ax.set_ylabel("Ocupación (%)"); ax.legend()
plt.show()

### ✏️ A14 · MINI-RETO — El número de control para la dirección

**Tu turno:** La dirección pide un **número de control**: ¿cuánto mejora (en %) el mejor modelo respecto al mejor baseline? Guarda en `mejor_baseline` el menor MAE entre los tres baselines y en `mejora_pct` el porcentaje de mejora de `mae_sar` respecto a él (`100 * (baseline - modelo) / baseline`, redondeado a 1 decimal; puede ser negativo). Escribe además en `conclusion` una frase (texto) para el cliente.

**Debe salir:** una frase de conclusión (`El modelo SARIMAX se equivoca de media 3.2 puntos de ocupación a 7 días, un 28.5% respecto al mejor método simple.`) y ✔ Mini-reto correcto

Cambia los `...` y pulsa ▶.

🧠 **Autoejercicio:** cuando salga ✔, responde (respuestas en la Parte B, A14).

1. ¿Cuánto mejora SARIMAX respecto al mejor baseline?
2. ¿Por qué la fórmula puede dar un porcentaje negativo?
3. ¿Qué debe incluir una buena frase para el cliente?

In [ ]:
# pista: min de los tres MAE de baseline; la fórmula de mejora está en el enunciado
mejor_baseline = ...
mejora_pct = ...
conclusion = "..."   # una frase para el cliente

assert abs(mejor_baseline - min(mae_ing, mae_est, mae_media4)) < 1e-9, "mejor_baseline es el mínimo de los tres MAE de baseline."
assert isinstance(conclusion, str) and len(conclusion) > 20, "Escribe una conclusión en una frase para el cliente."
print(conclusion)
print("✔ Mini-reto correcto")

# Parte B · Explicación de cada celda y de cada resultado

Cada apartado lleva el **mismo número** que su celda de la Parte A (A1, A2…). Si una celda no te sale, mira su «Si falla».

#### A1 · Preparación — Preparar las herramientas

- **Qué hace:**
    - Carga `numpy`, `pandas`, `matplotlib` y, de `statsmodels`, la descomposición estacional (`seasonal_decompose`) y el modelo `SARIMAX`.
    - `warnings.filterwarnings("ignore")` oculta avisos que no afectan a los resultados; `np.random.seed(SEMILLA)` fija la semilla y `plt.rcParams.update(...)` da un estilo común a los gráficos.
- **Qué debe salir:**
    - `Listo ✔`.
- **Cómo leerlo:**
    - No hay que entender esta celda: solo ejecutarla **la primera**, y otra vez si reinicias el entorno.
- **Si falla:**
    - `NameError: name 'pd' is not defined` → no has ejecutado esta celda. Ejecútala y repite la que falló.
    - `ModuleNotFoundError: No module named 'statsmodels'` → fuera de Colab, instala `statsmodels` en tu entorno.

## 1. Mirar la serie

#### A2 · Ejemplo — Cargar la ocupación diaria

**Contexto.** La dirección de TurisData quiere saber cuánta ocupación tendrá el alojamiento la próxima semana para organizar personal y compras. Tenemos la ocupación diaria (%) de 2023 a 2025.

- **Qué hace:**
    - `pd.read_csv(..., parse_dates=["fecha"]).set_index("fecha")` lee el archivo y usa la fecha como índice de la tabla.
    - `ocup.index.freq = "D"` indica que hay un dato por día (frecuencia diaria), algo que `statsmodels` necesita.
    - `ocup.shape` da su forma y `ocup.head()` muestra las 5 primeras filas.
- **Qué debe salir:**
    - `(1096, 4)` y una tabla con las columnas `ocupacion_pct`, `temperatura_c`, `festivo` y `precio_medio`, empezando el 2023-01-01.
- **Cómo leerlo:**
    - 1 096 días (tres años) con la ocupación en % y tres variables más. La columna `festivo` vale 1 en los festivos (más adelante será una variable conocida de antemano).
- **Si falla:**
    - `NameError: name 'dato' is not defined` → falta ejecutar la celda de preparación.

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *¿Cuántos días y columnas hay?* → **1 096** días y **4** columnas.
2. *¿Para qué sirve `set_index("fecha")`?* → Para que **la fecha sea el índice** de la tabla: así se ordena y se puede cortar por fechas.
3. *¿Qué indica `freq = "D"`?* → Que hay **un dato por día**, sin huecos.

</details>

#### A3 · Ejemplo — Dibujar la serie y sus patrones semanal y anual

**Idea.** Antes de modelar, se dibuja. Buscamos tres cosas: *tendencia* (¿sube o baja a largo plazo?),
*estacionalidad* (patrones que se repiten: la semana, el año) y *ruido*.

- **Qué hace:**
    - El primer gráfico dibuja la ocupación diaria de los tres años.
    - `por_dia` es la ocupación media por día de la semana (`groupby(ocup.index.dayofweek).mean()`) y `por_mes` la media por mes; se les pone etiquetas y se dibujan en dos gráficos de barras.
- **Qué debe salir:**
    - Un gráfico de línea con la serie y dos gráficos de barras (por día de la semana y por mes).
- **Cómo leerlo:**
    - **Estacionalidad semanal:** viernes, sábado y domingo rondan el 73 % y entre semana el 64–65 %. **Estacionalidad anual:** la ocupación es alta en invierno (78–79 % en enero y diciembre) y baja en verano y otoño (60 % en septiembre).
- **Si falla:**
    - `NameError: name 'ocup' is not defined` → falta el ejemplo anterior.

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *¿Qué días de la semana tienen más ocupación?* → Viernes, sábado y domingo: **≈ 73 %** frente a **64–65 %** entre semana.
2. *¿Qué mes tiene la ocupación más baja y cuál la más alta?* → La más baja, **septiembre** (≈ 60 %); la más alta, **enero** (≈ 79 %).
3. *¿Qué son la tendencia, la estacionalidad y el ruido?* → **Tendencia**: subida o bajada a largo plazo; **estacionalidad**: patrón que se repite (semana, año); **ruido**: el resto, sin patrón.

</details>

#### A4 · Ejercicio 1 — Día de más ocupación y mes de menos

- **Qué hace:**
    - `por_dia.idxmax()` devuelve la **etiqueta** del valor máximo (aquí, el nombre del día) y `por_mes.idxmin()` la del mínimo (el número de mes); `int(...)` lo convierte a entero.
- **Qué debe salir:**
    - `✔ Ejercicio 1 correcto`. `dia_max` vale `"sáb"` y `mes_min` vale `9`.
- **Si falla:**
    - `Revisa dia_max: el fin de semana es el que más ocupa` → `dia_max` no es `"vie"`, `"sáb"` o `"dom"`; quizá has usado `max()` (el valor) en lugar de `idxmax()` (la etiqueta).
    - `Revisa mes_min` → no has usado el mes de menor ocupación (`idxmin()` sobre `por_mes`).
    - `NameError: name 'por_dia' is not defined` → falta el ejemplo anterior.

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *¿Qué valen `dia_max` y `mes_min`?* → **"sáb"** (73,6 %) y **9** (septiembre, 60,1 %).
2. *¿Qué diferencia hay entre `max()` e `idxmax()`?* → `max()` da el **valor** máximo; `idxmax()` da **la etiqueta** (el índice) donde está.
3. *¿Por qué es útil conocer el patrón semanal?* → Porque permite **planificar personal y compras**: se sabe qué días hay más ocupación.

</details>

<details><summary>✅ Comprueba tu código: solución de referencia (solo después de intentarlo)</summary>

```python
dia_max = por_dia.idxmax()
mes_min = int(por_mes.idxmin())
```

**Tu código puede ser distinto y valer**: lo que importa es que salga ✔ y que los valores coincidan con los de «Qué debe salir».

</details>

## 2. Partición temporal (¡sin barajar!)

#### A5 · Ejemplo — Partir entrenamiento y prueba por fecha

**Idea.** Con datos de tabla partimos al azar. Con series **no**: el modelo no puede ver el futuro.
Entrenamos con el pasado y evaluamos con los últimos días. Aquí reservamos los **últimos 91 días** (13 semanas)
como test. Es la misma idea de "sin fuga de datos" que ya conoces, aplicada al tiempo.

- **Qué hace:**
    - `DIAS_TEST = 91` reserva los últimos 91 días (13 semanas) como prueba.
    - `serie` es la columna de ocupación; `serie.iloc[:-DIAS_TEST]` toma todo menos los últimos 91 días (`entrenamiento`) y `serie.iloc[-DIAS_TEST:]` los últimos 91 (`prueba`).
    - Los `print` muestran el rango de fechas y el número de días de cada parte.
- **Qué debe salir:**
    - `Entrenamiento: 2023-01-01 → 2025-10-01 1005` y `Prueba       : 2025-10-02 → 2025-12-31 91`.
- **Cómo leerlo:**
    - El entrenamiento llega hasta el 1 de octubre de 2025 y la prueba cubre el último trimestre: **el modelo nunca ve el futuro** al entrenar.
- **Si falla:**
    - `NameError: name 'ocup' is not defined` → falta ejecutar los ejemplos anteriores.

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *¿Cuántos días hay en entrenamiento y en prueba?* → **1 005** y **91**.
2. *¿Por qué no se parte al azar?* → Porque el modelo **no puede ver el futuro**: partir al azar mezclaría días posteriores con anteriores (fuga de datos).
3. *¿Qué usa `iloc[-DIAS_TEST:]`?* → **Posiciones** desde el final: los últimos 91 días de la serie.

</details>

## 3. Baselines: el rival que hay que batir

#### A6 · Ejemplo — Los dos primeros baselines (ingenuo y estacional)

**Idea.** Un modelo complejo solo vale la pena si gana a algo muy simple.
- **Ingenuo**: mañana será igual que hoy (repetimos el último valor conocido).
- **Estacional ingenuo**: el lunes que viene será como el último lunes (repetimos la última semana).

**Cómo evaluamos.** Imitamos el uso real: cada semana predecimos los **7 días siguientes** con los datos
conocidos hasta ese momento. Repetimos en las 13 semanas del test y medimos el **error absoluto medio (MAE)**,
en puntos de ocupación.

- **Qué hace:**
    - `origenes` son las posiciones donde empieza cada una de las 13 semanas de test.
    - `mae(real, pred)` calcula el error absoluto medio; `evaluar_semanal(predictor)` simula el uso real: para cada semana de prueba pide al `predictor` los 7 días siguientes usando solo la historia anterior y calcula el MAE total.
    - `pred_ingenuo` repite el último valor conocido y `pred_estacional` repite la última semana (`historia.iloc[-7:]`).
- **Qué debe salir:**
    - `MAE ingenuo    : 5.95 puntos` y `MAE estacional : 4.46 puntos`.
- **Cómo leerlo:**
    - Copiar el último día se equivoca de media 5,95 puntos de ocupación; copiar la última semana, 4,46: **la estacionalidad semanal ya ayuda**. Este es el listón que debe superar cualquier modelo más complejo.
- **Si falla:**
    - `NameError: name 'serie' is not defined` → falta el ejemplo de la partición temporal.

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *¿Qué MAE tienen el baseline ingenuo y el estacional?* → **5,95** y **4,46** puntos de ocupación.
2. *¿Cómo se imita el uso real en `evaluar_semanal`?* → Cada semana se predicen **los 7 días siguientes** con los datos conocidos hasta ese momento, durante las 13 semanas de prueba.
3. *¿Por qué se empieza con baselines?* → Porque un modelo complejo **solo vale la pena si gana a algo muy simple**.

</details>

#### A7 · Ejercicio 2 — Baseline: media de las últimas 4 semanas

- **Qué hace:**
    - `historia.iloc[-28:].values.reshape(4, 7)` toma las últimas 28 observaciones y las ordena en una matriz de 4 filas (semanas) por 7 columnas (días de la semana).
    - `.mean(axis=0)` hace la media de cada columna: la media de las últimas 4 semanas para cada día de la semana (7 valores).
    - `mae_media4, p_media4 = evaluar_semanal(pred_media4)` calcula su MAE y las comprobaciones verifican que devuelve 7 valores y que gana al ingenuo.
- **Qué debe salir:**
    - `MAE media 4 semanas: 4.45` y `✔ Ejercicio 2 correcto`.
- **Cómo leerlo:**
    - La media de 4 semanas (4,45) apenas mejora a la estacional (4,46): ambos baselines son muy parecidos.
- **Si falla:**
    - `pred_media4 debe devolver 7 valores` → la media no se hace por columnas (`axis=0`) o el `reshape` no es `(4, 7)`.
    - `La media de 4 semanas debería batir al ingenuo` → revisa el `reshape` y el eje de la media.
    - `NameError: name 'evaluar_semanal' is not defined` → falta el ejemplo anterior.

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *¿Cuánto vale el MAE de la media de 4 semanas y cómo se compara?* → **4,45**: casi igual que el estacional (4,46) y mejor que el ingenuo (5,95).
2. *¿Qué hace `reshape(4, 7)` y por qué `axis=0`?* → Ordena 28 días en **4 semanas × 7 días**; `axis=0` promedia **cada día de la semana** a lo largo de las 4 semanas.
3. *¿Por qué promediar 4 semanas en vez de copiar solo la última?* → Es **más estable**: un día raro de la última semana pesa menos.

</details>

<details><summary>✅ Comprueba tu código: solución de referencia (solo después de intentarlo)</summary>

```python
def pred_media4(historia, o):
    ultimas = historia.iloc[-28:].values.reshape(4, 7)
    return ultimas.mean(axis=0)
```

**Tu código puede ser distinto y valer**: lo que importa es que salga ✔ y que los valores coincidan con los de «Qué debe salir».

</details>

## 4. Descomposición: tendencia + estacionalidad + resto

#### A8 · Ejemplo — Descomponer la serie de entrenamiento

**Idea.** Una serie se puede ver como suma de piezas: `serie = tendencia + estacionalidad + resto`.
`seasonal_decompose` estima la parte que se repite cada 7 días y una tendencia suavizada.
Lo que queda (el *resto*) es lo que ningún patrón explica. Usamos solo el entrenamiento.

- **Qué hace:**
    - `seasonal_decompose(entrenamiento, model="additive", period=7)` separa la serie en tendencia, estacionalidad semanal y resto (`desc`).
    - Se dibujan los cuatro componentes (observada, tendencia, estacionalidad y resto) y se imprime el efecto semanal de los primeros 7 días.
- **Qué debe salir:**
    - Cuatro gráficos apilados y el efecto semanal: `2023-01-01  4.9`, `-3.3`, `-4.2`, `-3.2`, `-4.2`, `4.8` y `5.2` (el 1 de enero de 2023 fue domingo).
- **Cómo leerlo:**
    - El efecto semanal es de ≈ +5 puntos los fines de semana (viernes, sábado y domingo) y de entre −3 y −4 entre semana: **los fines de semana suben, los días laborables bajan**. El resto es lo que ningún patrón explica.
- **Si falla:**
    - `NameError: name 'entrenamiento' is not defined` → falta el ejemplo de la partición temporal.

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *¿Cuánto vale el efecto semanal de un fin de semana?* → Unos **+5 puntos** (4,8 a 5,2) sobre la media.
2. *¿Y entre semana?* → Entre **−3 y −4 puntos**.
3. *¿Qué representa el «resto»?* → **Lo que no explican la tendencia ni la estacionalidad**: el ruido.

</details>

#### A9 · Ejercicio 3 — Amplitud semanal y resto

- **Qué hace:**
    - `desc.seasonal.max() - desc.seasonal.min()` es la diferencia entre el mayor y el menor efecto semanal; `round(..., 1)` la deja con 1 decimal (`amplitud_semanal`).
    - `desc.resid.dropna().std()` es la desviación típica del resto sin los valores vacíos (los extremos de la serie no tienen resto; `resto_std`).
- **Qué debe salir:**
    - `✔ Ejercicio 3 correcto`. `amplitud_semanal` vale **9.4** (5,2 − (−4,2)) y `resto_std` ≈ **3.67**.
- **Cómo leerlo:**
    - La brecha entre un fin de semana y un día laborable es de ≈ 9 puntos de ocupación. La desviación típica del resto (3,67) es mucho menor que la de la serie original (9,61): **la estacionalidad semanal y la tendencia explican la mayor parte de la variación**.
- **Si falla:**
    - `amplitud_semanal debería rondar la brecha ... (~9 puntos)` → no estás restando el mínimo al máximo del componente `seasonal`.
    - `resto_std debe ser menor que la desviación de la serie original` → has calculado la desviación de la serie y no la del resto (`desc.resid`).
    - `NameError: name 'desc' is not defined` → falta el ejemplo anterior.

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *¿Cuánto valen `amplitud_semanal` y `resto_std`?* → **9,4** puntos y **≈ 3,67**.
2. *¿Por qué se usa `dropna()` con el resto?* → Porque `seasonal_decompose` deja **valores vacíos (NaN)** en los extremos, donde no puede calcular la tendencia.
3. *¿Qué indica que `resto_std` (3,67) sea mucho menor que la desviación de la serie (9,61)?* → Que **tendencia y estacionalidad explican la mayor parte** de la variación.

</details>

<details><summary>✅ Comprueba tu código: solución de referencia (solo después de intentarlo)</summary>

```python
amplitud_semanal = round(float(desc.seasonal.max() - desc.seasonal.min()), 1)
resto_std = float(desc.resid.dropna().std())
```

**Tu código puede ser distinto y valer**: lo que importa es que salga ✔ y que los valores coincidan con los de «Qué debe salir».

</details>

## 5. ARIMA y SARIMAX

#### A10 · Ejemplo — Ajustar un modelo SARIMAX con festivos y Fourier

**Idea sin fórmulas.**
- **AR (autorregresivo)**: el valor de hoy depende de los de días anteriores.
- **I (integrado)**: se trabaja con *diferencias* para quitar tendencias.
- **MA (media móvil)**: el valor depende de los errores recientes del modelo.
- **S (estacional)**: lo mismo pero mirando "hace 7 días".
- **X (exógenas)**: variables externas que conocemos de antemano, como los **festivos**.

Como la estacionalidad anual (invierno alto, verano bajo) es larga para SARIMAX, la representamos con
dos ondas seno/coseno del año (**términos de Fourier**), que son calendario y por tanto conocidos en el futuro.
**Cuidado:** no usamos la temperatura como variable, porque no la conoceríamos con antelación (sería *fuga*).

- **Qué hace:**
    - `exogenas(indice)` construye las variables externas conocidas de antemano: dos ondas seno/coseno del año (términos de Fourier, `sin1`, `cos1`, `sin2`, `cos2`) y los festivos.
    - `SARIMAX(entrenamiento, exog=X_ent, order=(1, 0, 1), seasonal_order=(1, 1, 1, 7), ...)` define el modelo (AR 1, MA 1 y parte estacional semanal) y `.fit(...)` lo ajusta (`res`).
    - `res.summary().tables[1]` imprime la tabla de coeficientes.
- **Qué debe salir:**
    - Una tabla de coeficientes (`sin1`, `cos1`, `sin2`, `cos2`, `festivo`, `ar.L1`, `ma.L1`, `ar.S.L7`, `ma.S.L7`, `sigma2`). Tarda unos segundos.
- **Cómo leerlo:**
    - El coeficiente de `festivo` es **+9,2**: un festivo suma unos 9 puntos de ocupación (con un valor `P>|z|` de 0,000, es decir, claramente distinto de 0). `cos1` (+8,0) recoge el patrón anual (alto en invierno). El término `ar.S.L7` es casi 0 (0,003, `P>|z|` = 0,52) y no aporta.
- **Si falla:**
    - `NameError: name 'entrenamiento' is not defined` → falta el ejemplo de la partición temporal.

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *¿Cuánto suma un festivo según el modelo?* → Unos **9,2 puntos** de ocupación.
2. *¿Por qué se usan términos de Fourier en vez de una estacionalidad anual de 365 días?* → Porque una estacionalidad tan larga es **pesada para SARIMAX**; dos ondas seno/coseno del año la representan con pocos parámetros, y son calendario (se conocen en el futuro).
3. *¿Por qué no se usa la temperatura como variable?* → Porque **no se conoce con antelación**: usarla sería *fuga de datos*.

</details>

#### A11 · Ejemplo — Predecir semana a semana con SARIMAX

**Cómo usarlo semana a semana.** No reajustamos el modelo cada semana (sería lento): reutilizamos los
parámetros aprendidos y solo le damos la historia nueva con `res.apply(...)`. Es el mismo proceso que en producción:
cada semana se actualiza la historia y se pide la previsión de 7 días.

- **Qué hace:**
    - `pred_sarimax(historia, o)` aplica el modelo ya ajustado a la historia nueva con `res.apply(historia, exog=..., refit=False)` (sin reajustar los parámetros) y pide `forecast(HORIZONTE, exog=...)` para los 7 días siguientes.
    - `evaluar_semanal(pred_sarimax)` repite el proceso en las 13 semanas de prueba y devuelve el MAE (`mae_sar`).
- **Qué debe salir:**
    - `MAE SARIMAX: 3.18   (estacional ingenuo: 4.46, media 4 sem.: 4.45)`.
- **Cómo leerlo:**
    - SARIMAX se equivoca de media 3,18 puntos frente a 4,45–4,46 de los mejores baselines: **gana con claridad** (≈ 28 % menos de error).
- **Si falla:**
    - `NameError: name 'res' is not defined` → falta el ejemplo anterior.

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *¿Cuánto vale el MAE de SARIMAX y cómo se compara con los baselines?* → **3,18** frente a **4,45–4,46** (mejores baselines).
2. *¿Por qué no se reajusta el modelo cada semana?* → Porque sería **lento**: se reutilizan los parámetros y solo se le da la historia nueva (`refit=False`), como en producción.
3. *¿Qué horizonte de previsión se usa?* → **7 días** (`HORIZONTE`).

</details>

#### A12 · Ejercicio 4 — SARIMAX sin exógenas

- **Qué hace:**
    - `res2 = SARIMAX(entrenamiento, order=(1, 0, 1), seasonal_order=(1, 1, 1, 7), ...).fit(...)` ajusta el modelo sin festivos ni Fourier.
    - `pred_sarimax_simple(historia, o)` hace la previsión de 7 días con `res2.apply(historia, refit=False).forecast(HORIZONTE)` (sin `exog`).
    - `mae_sin_exog, _ = evaluar_semanal(pred_sarimax_simple)` calcula su MAE; el `_` ignora las predicciones.
- **Qué debe salir:**
    - `MAE SARIMAX sin exógenas: 3.93  |  con exógenas: 3.18` y `✔ Ejercicio 4 correcto`.
- **Cómo leerlo:**
    - Sin festivos ni Fourier el error sube de 3,18 a 3,93: **las variables conocidas de antemano aportan ≈ 0,75 puntos**. Aun así, el modelo sin exógenas (3,93) sigue ganando a los baselines (4,45).
- **Si falla:**
    - `mae_sin_exog debe ser un MAE razonable` → `mae_sin_exog` no está definido o no es el MAE (por ejemplo, has guardado la tupla).
    - `TypeError ... exog` → en `res2.apply(...)` y en `forecast(...)` no hay que pasar `exog`.
    - `NameError: name 'evaluar_semanal' is not defined` → falta el ejemplo de baselines.

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *¿Cuánto vale el MAE sin exógenas y cuánto con ellas?* → **3,93** y **3,18**.
2. *¿Qué aportan los festivos y los términos de Fourier?* → Reducen el error en **≈ 0,75 puntos**.
3. *¿Gana el modelo sin exógenas a los baselines?* → **Sí**: 3,93 frente a 4,45.

</details>

<details><summary>✅ Comprueba tu código: solución de referencia (solo después de intentarlo)</summary>

```python
res2 = SARIMAX(entrenamiento, order=(1, 0, 1), seasonal_order=(1, 1, 1, 7),
               enforce_stationarity=False, enforce_invertibility=False).fit(disp=False, maxiter=200)

def pred_sarimax_simple(historia, o):
    return res2.apply(historia, refit=False).forecast(HORIZONTE).values

mae_sin_exog, _ = evaluar_semanal(pred_sarimax_simple)
```

**Tu código puede ser distinto y valer**: lo que importa es que salga ✔ y que los valores coincidan con los de «Qué debe salir».

</details>

## 6. Comparar y mirar los errores

#### A13 · Ejemplo — Tabla comparativa y gráfico de las previsiones

Una tabla y un gráfico: ¿gana el modelo sofisticado a los baselines? Si la ganancia es pequeña, a veces
**el baseline es la respuesta correcta para el cliente** (más simple, más fácil de explicar).

- **Qué hace:**
    - `comparativa` reúne los cinco MAE en una serie ordenada de menor a mayor error y la imprime.
    - El gráfico dibuja la ocupación real del trimestre de prueba (en negro) y las previsiones a 7 días del baseline estacional y de SARIMAX.
- **Qué debe salir:**
    - La tabla: `SARIMAX + festivos + Fourier 3.18` · `SARIMAX sin exógenas 3.93` · `Media 4 semanas 4.45` · `Estacional (t-7) 4.46` · `Ingenuo 5.95`, y un gráfico con tres líneas.
- **Cómo leerlo:**
    - El orden de los modelos es el esperado: cuanto más información usa el modelo, menos error. La línea de SARIMAX sigue mejor a la real que la del estacional. Si la ganancia hubiera sido pequeña, **el baseline sería la respuesta correcta** para el cliente (más simple y fácil de explicar).
- **Si falla:**
    - `NameError: name 'mae_sin_exog' is not defined` → falta el ejercicio 4.

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *¿Qué modelo tiene menos error y cuál más?* → Menos: **SARIMAX + festivos + Fourier** (3,18). Más: el **ingenuo** (5,95).
2. *¿Cuándo sería mejor quedarse con un baseline?* → Cuando el modelo sofisticado **gana poco**: el baseline es más simple, más fácil de explicar y de mantener.
3. *¿Por qué se dibuja también la serie real?* → Para **ver dónde falla** cada previsión, no solo cuánto error medio tiene.

</details>

## 7. Mini-reto integrador

#### A14 · Mini-reto — El número de control para la dirección

- **Qué hace:**
    - `mejor_baseline = min(mae_ing, mae_est, mae_media4)` es el menor error entre los tres baselines (4,45).
    - `mejora_pct = round(100 * (mejor_baseline - mae_sar) / mejor_baseline, 1)` es el porcentaje en que SARIMAX reduce ese error (28,5 %).
    - `conclusion` es una frase (texto) para el cliente que usa esas cifras; la comprobación la imprime.
- **Qué debe salir:**
    - `El modelo SARIMAX se equivoca de media 3.2 puntos de ocupación a 7 días, un 28.5% respecto al mejor método simple.` (la tuya puede estar redactada de otro modo) y `✔ Mini-reto correcto`.
- **Cómo leerlo:**
    - Hay que decir **cuánto mejora** y **en qué unidades**, y un 28,5 % menos de error respecto al mejor método simple es una mejora que justifica el modelo. Ojo con el signo: si SARIMAX fuera peor, `mejora_pct` saldría negativo.
- **Si falla:**
    - `mejor_baseline es el mínimo de los tres MAE de baseline` → has usado otro valor (por ejemplo, la media) o has incluido `mae_sar`.
    - `Escribe una conclusión en una frase para el cliente` → `conclusion` no es un texto de más de 20 caracteres.
    - `NameError: name 'mae_sar' is not defined` → falta el ejemplo de SARIMAX.

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *¿Cuánto mejora SARIMAX respecto al mejor baseline?* → **28,5 %** (de 4,45 a 3,18 puntos de error).
2. *¿Por qué la fórmula puede dar un porcentaje negativo?* → Si el modelo tuviera **más error** que el baseline, `baseline − modelo` sería negativo: el modelo empeoraría.
3. *¿Qué debe incluir una buena frase para el cliente?* → El **error medio en puntos de ocupación**, el **horizonte** (7 días) y **cuánto mejora** respecto a la alternativa simple.

</details>

## Preguntas de reflexión
1. ¿Por qué barajar las filas antes de partir train/test sería un error en una serie temporal?
2. Si el mejor baseline queda muy cerca de SARIMAX, ¿qué recomendarías al cliente y por qué?
3. ¿Qué otra variable exógena podría ayudar y **cuándo** la conoceríamos? (pista: ¿se conoce el futuro?)
4. El modelo prevé bien el día "normal"; ¿qué pasaría con un evento extraordinario (un temporal, un congreso)?

<details><summary>Ver respuestas (después de pensarlo)</summary>

1. Porque mezclaría el **pasado y el futuro**: el modelo entrenaría con días posteriores a algunos del test (fuga de datos) y la evaluación saldría demasiado optimista. Además, los días vecinos se parecen mucho entre sí, así que el test quedaría «contaminado» por el entrenamiento.
2. Recomendar el **baseline**: es más simple, más fácil de explicar y de mantener, y una mejora pequeña no compensa la complejidad. (Aquí no es el caso: SARIMAX mejora un 28,5 %.)
3. Variables **conocidas de antemano**: festivos, calendario escolar, eventos programados (congresos, conciertos) o las reservas ya confirmadas. La previsión meteorológica a 7 días también se conoce con antelación (con error), pero **la temperatura real del día no**.
4. Con un evento extraordinario el modelo **falla**: no está en la historia ni en las variables, así que su previsión sería muy mala ese día. Si el evento se conoce antes, se puede añadir como variable; si no, hace falta revisión humana o intervalos de incertidumbre.

</details>

## Resumen: qué has aprendido
- Las series se parten **por fecha** y se evalúan imitando el uso real (previsión a 7 días).
- Siempre hay que compararse con un **baseline**; a veces gana o casi.
- La descomposición separa tendencia, estacionalidad y resto; SARIMAX combina memoria, estacionalidad y variables conocidas.
- Evita usar como entrada datos que no tendrás en el momento de predecir.